# 00 - Conversão dos dados de origem (CSV -> pickle)

Converte os dois CSV do Kaggle em pickles, para as sessões seguintes abrirem os dados muito mais rápidamente

**O que este notebook faz**
1. Converte `accepted_2007_to_2018Q4.csv.gz` e `rejected_2007_to_2018Q4.csv.gz` (pasta `data/raw/`) em pickles (pasta `data/processed/`).
2. Verifica a cópia: número de linhas e colunas, tipos, formato dos valores.
3. Regista os números em `docs/dados_origem.md`, para o relatório.

**Decisões** (explicadas também em `src/lending/conversion.py`)
- Todas as colunas são lidas como **texto**: a conversão muda só o formato do ficheiro, não interpreta valores. Tipos e limpeza ficam para a Fase 2 (ELT).
- Só o **campo vazio** conta como valor em falta; textos como `"n/a"` ficam como estão.

**O que NÃO se faz aqui:** limpeza, exclusões, EDA. Este notebook não tira conclusões dos dados.

**Antes de correr:** a conversão do ficheiro *accepted* precisa de vários GB de RAM livre, por isso recomendamos fechar programas pesados.


In [1]:
# Setup: imports e caminhos do projeto
import gc
import platform
import sys
import time
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import numpy as np
import pandas as pd
import pyarrow

from lending import config
from lending.conversion import CsvToPickleConverter, DatasetSummary
from lending.storage import PickleStore

# Mostrar todas as colunas quando se imprime um DataFrame
pd.set_option("display.max_columns", None)

print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
      f"pyarrow {pyarrow.__version__} | numpy {np.__version__}")
print("Raiz do projeto:", config.PROJECT_ROOT)


Python 3.12.0 | pandas 3.0.6 | pyarrow 25.0.1 | numpy 2.5.3
Raiz do projeto: C:\Users\joaop\PycharmProjects\ProjetoCD-LendingClub


## 1. Conversão

Cada conversão corre **uma única vez**. Se o pickle já existir é saltada (se quiser refazer basta colocar este parametro no convert: `convert(overwrite=True)`).

- *accepted*: 151 colunas -> blocos de 100 000 linhas.
- *rejected*: 9 colunas -> blocos de 1 000 000 linhas.

Se der `MemoryError` ou o computador ficar muito lento, basta reduzir o `chunksize`.


In [2]:
accepted_converter = CsvToPickleConverter(
    csv_path=config.ACCEPTED_CSV,
    pickle_path=config.ACCEPTED_RAW_PKL,
    chunksize=100_000,
)
accepted_converter.convert()

# Libertar a memória usada durante a conversão antes da próxima
_ = gc.collect()


accepted_raw.pkl já existe - conversão saltada. Para refazer: convert(overwrite=True).


In [3]:
rejected_converter = CsvToPickleConverter(
    csv_path=config.REJECTED_CSV,
    pickle_path=config.REJECTED_RAW_PKL,
    chunksize=1_000_000,
)
rejected_converter.convert()

_ = gc.collect()


rejected_raw.pkl já existe - conversão saltada. Para refazer: convert(overwrite=True).


## 2. Verificação

Para cada pickle:
1. Abrir e medir o tempo (o esperado é que seja em segundos).
2. Confirmar o número de colunas esperado (151 e 9) e que todas são texto.
3. Ver o **formato** dos valores numa amostra aleatória com semente.
   *Não é uma amostra para tirar conclusões*, serve só para planear o parsing da Etapa 2
   É aleatória porque o enunciado proíbe usar as primeiras linhas como amostra
4. Ver as últimas linhas do ficheiro: é comum ficheiros exportados terminarem com linhas de totais ou rodapé. Se existirem ficam na cópia e saem na limpeza (Etapa 2).

Os ficheiros são verificados um de cada vez, libertando a memória entre eles.


In [4]:
# accepted: abrir e medir
t0 = time.perf_counter()
accepted = PickleStore.load(config.ACCEPTED_RAW_PKL, expected_type=pd.DataFrame)
load_seconds = time.perf_counter() - t0

accepted_summary = DatasetSummary.from_dataframe(
    "accepted", accepted, config.ACCEPTED_CSV, config.ACCEPTED_RAW_PKL, load_seconds
)
print(accepted_summary)

# verificações
assert accepted.shape[1] == config.EXPECTED_COLUMNS["accepted"], "Número de colunas inesperado"
assert (accepted.dtypes == accepted.dtypes.iloc[0]).all(), "Há colunas que não são texto"
print("OK: colunas e tipos como esperado |", accepted.dtypes.iloc[0])


accepted: 2,260,701 linhas x 151 colunas | CSV 393 MB -> pickle 4,107 MB | memória 4,107 MB | abriu em 8.9s
OK: colunas e tipos como esperado | str


In [5]:
# Formato dos valores (amostra aleatória com semente - não serve para conclusões)
format_columns = ["id", "issue_d", "loan_amnt", "term", "int_rate", "grade",
                  "emp_length", "annual_inc", "dti", "revol_util",
                  "earliest_cr_line", "loan_status"]
format_columns = [c for c in format_columns if c in accepted.columns]

accepted[format_columns].sample(5, random_state=config.RANDOM_SEED)


,id,issue_d,loan_amnt,term,int_rate,grade,emp_length,annual_inc,dti,revol_util,earliest_cr_line,loan_status
392949,39651438,Feb-2015,32000.0,60 months,10.49,B,10+ years,120000.0,24.05,57.8,Oct-1981,Current
1273506,16411620,May-2014,9600.0,36 months,12.99,C,NaN,21900.0,10.03,38.9,Apr-2001,Fully Paid
324024,45122316,Apr-2015,4000.0,36 months,6.68,A,4 years,83000.0,19.53,17.2,Sep-2003,Fully Paid
2066630,125356772,Dec-2017,6025.0,36 months,10.91,B,10+ years,52000.0,9.16,12.8,Jun-2005,Fully Paid
477199,128490686,Feb-2018,25000.0,60 months,26.3,E,10+ years,65000.0,36.26,24.7,Jul-1999,Current


In [6]:
# Fim do ficheiro
accepted[["id", "loan_amnt", "issue_d", "loan_status"]].tail(3)


,id,loan_amnt,issue_d,loan_status
2260698,88215728,14000.0,Oct-2016,Current
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN
2260700,Total amount funded in policy code 2: 521953170,NaN,NaN,NaN


In [7]:
# Libertar memória antes de abrir o próximo ficheiro
del accepted
_ = gc.collect()


In [8]:
# rejected: abrir e medir
t0 = time.perf_counter()
rejected = PickleStore.load(config.REJECTED_RAW_PKL, expected_type=pd.DataFrame)
load_seconds = time.perf_counter() - t0

rejected_summary = DatasetSummary.from_dataframe(
    "rejected", rejected, config.REJECTED_CSV, config.REJECTED_RAW_PKL, load_seconds
)
print(rejected_summary)

assert rejected.shape[1] == config.EXPECTED_COLUMNS["rejected"], "Número de colunas inesperado"
assert (rejected.dtypes == rejected.dtypes.iloc[0]).all(), "Há colunas que não são texto"
print("OK: colunas e tipos como esperado |", rejected.dtypes.iloc[0])
print("Colunas:", rejected.columns.tolist())


rejected: 27,648,741 linhas x 9 colunas | CSV 255 MB -> pickle 3,545 MB | memória 3,545 MB | abriu em 5.4s
OK: colunas e tipos como esperado | str
Colunas: ['Amount Requested', 'Application Date', 'Loan Title', 'Risk_Score', 'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length', 'Policy Code']


In [9]:
# Formato dos valores (amostra aleatória com semente) e fim do ficheiro
display(rejected.sample(5, random_state=config.RANDOM_SEED))
display(rejected.tail(3))


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
23949778,15000.0,2015-10-29,debt_consolidation,NaN,54.58%,761xx,TX,< 1 year,0.0
20321967,10000.0,2017-10-28,Credit card refinancing,605.0,28.71%,076xx,NJ,NaN,0.0
11092152,10000.0,2018-03-09,Debt consolidation,NaN,58.06%,087xx,NJ,< 1 year,0.0
26499034,7000.0,2016-10-18,Debt consolidation,571.0,6.66%,731xx,OK,< 1 year,0.0
3616603,5000.0,2017-02-07,other,NaN,2.23%,537xx,WI,< 1 year,0.0


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
27648738,1200.0,2016-12-31,Other,686.0,10.26%,914xx,CA,< 1 year,0.0
27648739,25000.0,2016-12-31,debt_consolidation,NaN,17.71%,880xx,NM,< 1 year,0.0
27648740,15000.0,2016-12-31,Business,684.0,10.58%,113xx,NY,< 1 year,0.0


In [10]:
del rejected
_ = gc.collect()


## 3. Registo para o relatório

Escreve `docs/dados_origem.md` com a tabela dos dois ficheiros e as versões usadas.

Preenche a data em que foi descarregada os dados do Kaggle.


In [11]:
DOWNLOAD_DATE = "2026-10-14"   # data do download do Kaggle

summaries = [accepted_summary, rejected_summary]
out = DatasetSummary.write_markdown(summaries, config.DOCS_DIR / "dados_origem.md",
                                    download_date=DOWNLOAD_DATE)
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

pd.DataFrame([s.as_dict() for s in summaries]).set_index("name").round(1)


Escrito: docs\dados_origem.md


,n_rows,n_columns,csv_mb,pickle_mb,memory_mb,load_seconds
name,,,,,,
accepted,2260701,151,392.6,4107.0,4106.9,8.9
rejected,27648741,9,255.5,3544.9,3544.9,5.4
